# Fase 4 — RAG com fontes e consulta estruturada

**Problema desta fase:** responder "onde foi dito?" citando a mensagem original e responder "quantas mensagens…?" com número **exato**, sem confundir as duas coisas.

**Conceitos da disciplina demonstrados:** índice vetorial FAISS com embeddings, recuperação top-k com fonte, text-to-query com plano tipado e SQL parametrizado, diagnóstico de falha de recuperação.

**Decisões principais**
- RAG serve para **localizar fonte**, nunca para contar ou calcular saldo (o top-k pode omitir mensagens).
- Contagens e listas usam **consulta estruturada**: o LLM gera um plano tipado; Python valida e executa o `SELECT` parametrizado.
- Preparar o índice exige confirmação antes de enviar textos à OpenAI.

**Evidências reais (fora deste notebook):** [falha: agente não buscou a fonte](../evidencias/fase4_falha_busca_fonte_manual.json), [falha de ranking (Gama) com antes/depois](../evidencias/fase4_falha_ranking_gama.json), [reteste do text-to-query](../evidencias/fase4_text_to_query_reteste_manual.json). Guias: [indice_fontes_fase4.md](../docs/indice_fontes_fase4.md), [consulta_estruturada_fase4.md](../docs/consulta_estruturada_fase4.md).

> **Fixtures:** vetores e planos de consulta são fixtures locais; o diagnóstico de ranking reaplica a regra sobre escores **reais** salvos na evidência, sem chamar a API.

Notebooks de origem: `incrementos/10_indice_fontes_faiss.ipynb`, `incrementos/11_text_to_query_mensagens.ipynb`, `incrementos/12_diagnostico_ranking_rag.ipynb`.

---
*Origem desta parte: `incrementos/10_indice_fontes_faiss.ipynb`.*

## Fase 4 — índice FAISS de mensagens com fonte

Este notebook prepara um índice a partir da conversa **fictícia**, usando vetores de fixture. A busca carrega o índice já salvo; não reindexa a cada pergunta. Não há chamada de API nem interpretação semântica por LLM nesta demonstração.


In [1]:
from pathlib import Path
import sys, tempfile
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/secretaria').is_dir())
sys.path.insert(0,str(raiz / 'src'))
from secretaria.banco import Banco, Contexto
from secretaria.indice_fontes import IndiceFontes


In [2]:
class EmbeddingsFixture:
    def vetor(self,texto):
        texto = texto.lower()
        return [1.0 if 'alfa' in texto else 0.0, 1.0 if 'beta' in texto else 0.0, 0.1]
    def embed_documents(self,textos):
        return [self.vetor(texto) for texto in textos]
    def embed_query(self,texto):
        return self.vetor(texto)


### Preparar, carregar e conferir a fonte
A resposta recuperada contém `mensagem_id` e o texto original do SQLite. Ela é evidência de onde algo foi dito; não comprova pagamento ou saldo.


In [3]:
with tempfile.TemporaryDirectory() as pasta:
    with Banco(Path(pasta)/'demo.sqlite3') as banco:
        banco.preparar_demo()
        ana, bruno = Contexto('demo','ana'), Contexto('demo','bruno')
        corpus = banco.selecionar_corpus(ana,'Demonstração fictícia')
        banco.importar(ana,corpus,raiz/'dados/ficticios/conversa_empresa.txt')
        indice = IndiceFontes(banco,Path(pasta)/'indices')
        fixture = EmbeddingsFixture()
        preparo = indice.preparar(ana,corpus,fixture,'fixture_manual_v1')
        assert preparo['estado'] == 'preparado'
        assert indice.preparar(ana,corpus,fixture,'fixture_manual_v1')['estado'] == 'atual'
        achados = indice.buscar(ana,corpus,'Alfa',fixture,'fixture_manual_v1',limite=2)
        assert 'Alfa' in achados[0]['texto']
        print('Versão:',preparo['versao_corpus'][:12], 'Fonte:',achados[0]['mensagem_id'])
        try:
            indice.buscar(bruno,corpus,'Alfa',fixture,'fixture_manual_v1')
        except PermissionError:
            print('Corpus privado: acesso de Bruno bloqueado')
        else:
            raise AssertionError('Falha de escopo')


Versão: c91959596d26 Fonte: 4
Corpus privado: acesso de Bruno bloqueado


> **Como ler o resultado:**
>
> - **Busca semântica (RAG) com FAISS**: a pergunta "Alfa" achou a mensagem certa e devolveu a **fonte** (`Fonte: 4` = mensagem 4).
> - `Versão`: identifica o conteúdo indexado; se as mensagens mudarem, o índice é refeito (no aplicativo aparece `'estado': 'atual'`).
> - `acesso de Bruno bloqueado`: o corpus é privado; outro usuário não busca nele.
> - Embeddings fixos aqui (sem API); no programa vêm da OpenAI (`text-embedding-3-small`).

### Uso no aplicativo
O menu 11 prepara o corpus selecionado com embeddings reais apenas após confirmação. No menu 9, a ferramenta `buscar_mensagens_fonte` carrega o índice existente e retorna fontes autorizadas. O notebook verifica a infraestrutura com fixture; a trajetória real do agente ainda exige teste manual.


---
*Origem desta parte: `incrementos/11_text_to_query_mensagens.ipynb`.*

## Text-to-query com plano tipado

Fixture local identificada: nenhuma chamada ao LLM. O modelo real teria de produzir o mesmo tipo de plano antes da validação.


In [4]:
import sys, tempfile
from pathlib import Path
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/secretaria').is_dir())
sys.path.insert(0, str(raiz / 'src'))
from secretaria.banco import Banco, Contexto
from secretaria.consulta_estruturada import executar_plano, PlanoConsultaMensagens


In [5]:
pergunta = 'Quantas mensagens de Bruno mencionam orçamento e Alfa?'
plano_fixture = PlanoConsultaMensagens(tipo='contagem', autor='Bruno', termos=['orçamento', 'Alfa'])
with tempfile.TemporaryDirectory() as pasta:
    with Banco(Path(pasta) / 'demo.sqlite3') as banco:
        banco.preparar_demo()
        contexto = Contexto('demo', 'ana')
        corpus_id = banco.selecionar_corpus(contexto, 'Demonstração fictícia')
        banco.importar(contexto, corpus_id, raiz / 'dados/ficticios/conversa_empresa.txt')
        resultado = executar_plano(banco, contexto, corpus_id, plano_fixture)
        print('Pergunta:', pergunta)
        print('Plano (FIXTURE LOCAL):', resultado['plano'])
        print('SQL permitido:', resultado['sql_executado'])
        print('Parâmetros:', resultado['parametros_sql'])
        print('Resultado:', resultado['total'])
        assert resultado['total'] == 1


Pergunta: Quantas mensagens de Bruno mencionam orçamento e Alfa?
Plano (FIXTURE LOCAL): {'tipo': 'contagem', 'autor': 'Bruno', 'termos': ['orçamento', 'Alfa'], 'data_inicio_local': None, 'data_fim_local_exclusiva': None, 'ordem': 'recente', 'limite': 20, 'motivo': None}
SQL permitido: SELECT COUNT(*) FROM mensagens_importadas WHERE empresa_id=? AND corpus_id=? AND LOWER(autor)=LOWER(?) AND LOWER(texto) LIKE LOWER(?) ESCAPE '\' AND LOWER(texto) LIKE LOWER(?) ESCAPE '\'
Parâmetros: ['demo', 1, 'Bruno', '%orçamento%', '%Alfa%']
Resultado: 1


> **Como ler o resultado:**
>
> - **Text-to-query**: para "quantas", não serve busca por semelhança; precisa de **número exato**.
> - `Plano`: o modelo só preenche um plano tipado (contagem, autor Bruno, termos orçamento e Alfa). Ele **não escreve SQL**.
> - `SQL permitido` + `Parâmetros`: o Python monta um SQL parametrizado e seguro a partir do plano.
> - `Resultado: 1`: uma mensagem do Bruno fala de orçamento e Alfa.

A contagem percorre as mensagens importadas autorizadas. Ela não calcula saldo nem comprova recebimento. Na aplicação, a OpenAI gera o plano tipado; Python continua sendo responsável por validar e executar o SELECT.


---
*Origem desta parte: `incrementos/12_diagnostico_ranking_rag.ipynb`.*

## Diagnóstico de ranking no RAG

Este notebook reproduz a reordenação com um registro de resultados obtidos anteriormente no índice real do corpus fictício. **Não chama a API neste notebook.** Veja a evidência JSON para os escores observados.


In [6]:
import json, sys
from pathlib import Path
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/secretaria').is_dir())
sys.path.insert(0, str(raiz / 'src'))
from secretaria.indice_fontes import reordenar_fontes_afirmativas
evidencia = json.loads((raiz / 'evidencias/fase4_falha_ranking_gama.json').read_text(encoding='utf-8'))
candidatos = [{'mensagem_id': item['mensagem_id'], 'texto': item['texto'], 'similaridade': item['similaridade']} for item in evidencia['resultado_vetorial_antes']]
depois, criterio = reordenar_fontes_afirmativas(evidencia['pergunta'], candidatos)
print('Antes:', [(item['mensagem_id'], item['similaridade']) for item in candidatos])
print('Depois:', [(item['mensagem_id'], item['posicao_vetorial']) for item in depois])
print('Critério:', criterio)
assert candidatos[0]['mensagem_id'] == 23
assert depois[0]['mensagem_id'] == 24


Antes: [(23, 0.8093), (24, 0.5637), (22, 0.5049)]
Depois: [(24, 2), (22, 3), (23, 1)]
Critério: afirmacao_antes_de_pergunta


> **Como ler o resultado:**
>
> - **Falha real corrigida**: perguntando o saldo da Gama, a busca vetorial pôs em 1º a M0023, que só **pergunta** o saldo (`0.8093`).
> - `Depois`: a regra reordena e põe em 1º a M0024, que **responde** (explica por que o saldo não pode ser calculado).
> - `Critério: afirmacao_antes_de_pergunta`: quando o pedido é por uma afirmação, mensagem que afirma vem antes de mensagem que pergunta.

M0023 pergunta pelo saldo; M0024 explica por que ele não pode ser calculado. A regra de reordenação considera a forma da mensagem apenas para pedidos por uma afirmação. Ainda é necessário conferir o conteúdo, pois uma frase declarativa pode ser irrelevante.
